# Research assistant: measure retrieval, filter by meaning *and* metadata

Two questions decide a RAG system's quality: *which retrieval mode fits my data?*
and *how do I combine semantic search with hard constraints?* Infino answers both
on **one** store — every retrieval mode (vector, BM25, hybrid) runs over the same
table, so comparing them is one line each, and structured filters push down to SQL.

This example (1) measures `recall@10` for vector, BM25, and hybrid on a labeled IR
benchmark, (2) filters semantic search by a metadata predicate, and (3) lets an
LLM turn a natural-language request into that filter automatically (self-query).
Parts 1–2 run **key-free**; part 3 needs an LLM key.

In [1]:
import sys
import shutil
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # examples/ root, where _shared/ lives

import infino
import pyarrow as pa
from langchain_infino import InfinoVectorStore, InfinoTranslator

from _shared.lc import MiniLMEmbeddings, chat_model
from _shared.embedding import DIM, METRIC
from _shared.loaders import load_ms_marco, load_arxiv

DB_DIR = "./research_data"
shutil.rmtree(DB_DIR, ignore_errors=True)  # start fresh each run

embeddings = MiniLMEmbeddings()
llm = chat_model()  # None when no key is set; only part 3 needs it
db = infino.connect(DB_DIR)
print("LLM (part 3):", "on" if llm else "off")

LLM (part 3): on


## Part 1 — which retrieval mode? Measure it.

BM25 matches exact terms; vector captures meaning; **hybrid** fuses both with
Reciprocal Rank Fusion. All three run over **one** Infino table —
`as_bm25_retriever`, `as_retriever`, `as_hybrid_retriever` — so you can measure
them head to head on your own data.

We index MS MARCO passages (each carries a ground-truth set of relevant passage
ids) and report `recall@10`.

In [2]:
passages, queries = load_ms_marco(n_queries=120)
marco = InfinoVectorStore.from_texts(
    [p["text"] for p in passages], embeddings,
    connection=db, table_name="passages", dim=DIM, metric=METRIC,
    ids=[str(p["pid"]) for p in passages],  # doc id == passage id, for scoring
)
print(f"indexed {len(passages)} passages, {len(queries)} labeled queries")

K = 10
retrievers = {
    "bm25": marco.as_bm25_retriever(k=K),
    "vector": marco.as_retriever(search_kwargs={"k": K}),
    "hybrid": marco.as_hybrid_retriever(k=K),
}


def recall_at_k(retriever, sample) -> float:
    total = 0.0
    for q in sample:
        got = {int(d.id) for d in retriever.invoke(q["query"]) if d.id is not None}
        relevant = set(q["relevant_pids"])
        total += len(got & relevant) / len(relevant)
    return total / len(sample)


sample = queries[:80]
for name, r in retrievers.items():
    print(f"recall@{K}  {name:>6}: {recall_at_k(r, sample):.3f}")

indexed 1031 passages, 120 labeled queries
recall@10    bm25: 0.900


recall@10  vector: 1.000


recall@10  hybrid: 1.000


**Read the numbers, don't assume.** Which mode wins depends on your data:
on these paraphrased queries the semantic signal is strong, so vector leads; on
keyword-heavy corpora (codes, SKUs, identifiers) BM25 and hybrid come into their
own. Because all three run on one store, you measure them in one line each and
pick what fits — hybrid is a solid default when the query mix is mixed or
unknown.

## Part 2 — semantic search with a metadata filter (key-free)

Promote metadata to scalar columns and Infino filters them as a SQL predicate
*pushed down* into the search. Here: papers about a topic, restricted to recent
years.

In [3]:
papers = load_arxiv(150)
years = [2019, 2021, 2022, 2023, 2024] * ((len(papers) // 5) + 1)
arxiv = InfinoVectorStore.from_texts(
    [p["abstract"] for p in papers], embeddings,
    metadatas=[{"title": p["title"], "year": y} for p, y in zip(papers, years)],
    connection=db, table_name="papers", dim=DIM, metric=METRIC,
    metadata_columns=[pa.field("year", pa.int64(), nullable=False)],
)
hits = arxiv.similarity_search(
    "representation learning", k=4, filter={"year": {"$gte": 2023}}
)
print("filter year>=2023 ->", sorted(d.metadata["year"] for d in hits))

filter year>=2023 -> [2023, 2024, 2024, 2024]


## Part 3 — self-query: let the LLM write the filter (needs an LLM key)

A `SelfQueryRetriever` uses the LLM to parse a natural-language request into a
structured query; `InfinoTranslator` turns that into Infino's SQL filter. So
*"papers about neural networks published since 2023"* becomes a semantic search
plus `year >= 2023`, with no hand-written predicate.

Requires `langchain-classic` and `lark` (see `requirements.txt`).

In [4]:
if llm:
    from langchain_classic.retrievers.self_query.base import SelfQueryRetriever
    from langchain_classic.chains.query_constructor.schema import AttributeInfo

    examples = [
        ("papers about transformers from 2021",
         {"query": "transformers", "filter": 'eq("year", 2021)'}),
        ("graph neural network work before 2020",
         {"query": "graph neural networks", "filter": 'lt("year", 2020)'}),
    ]
    self_query = SelfQueryRetriever.from_llm(
        llm, arxiv, "Machine learning paper abstracts",
        [AttributeInfo(name="year", description="publication year", type="integer")],
        structured_query_translator=InfinoTranslator(),
        chain_kwargs={"examples": examples},
    )
    out = self_query.invoke("papers about neural networks published since 2023")
    print("self-query ->", sorted(d.metadata["year"] for d in out))
else:
    print("skipped — set an LLM key to run self-query (see the README)")

self-query -> [2023, 2024, 2024, 2024]


## Takeaway

One Infino table let us **measure** three retrieval modes head to head, **filter**
semantic search by a SQL predicate, and let an LLM **write** that filter from plain
English. Pick the retrieval mode your numbers justify; reach for hybrid when the
query mix is mixed or unknown.